# RetailPulse AI Customer Analytics

## Notebook 01: Data Preparation & Feature Engineering

### Project Overview

RetailPulse AI Customer Analytics is an end-to-end machine learning project that analyzes historical retail transactions to generate actionable business insights.

This notebook focuses on preparing high-quality transactional data that will serve as the foundation for all downstream analytics and machine learning tasks.

### Objectives

- Load the Online Retail II dataset
- Assess data quality
- Clean and preprocess transactions
- Engineer business features
- Prepare datasets for customer analytics
- Export reusable processed datasets

### Outputs

- retail_cleaned.csv
- analysis_data.csv

## Project Directory Structure

```
RetailPulse-AI-Customer-Analytics/

│
├── data/
│   ├── raw/
│   │     online_retail_II.xlsx
│   │
│   └── processed/
│         retail_cleaned.csv
│         analysis_data.csv
│
├── notebooks/
│
├── models/
│
└── reports/
```

All notebooks will use these standardized project paths to ensure reproducibility.

In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.2f}".format)

# Plot settings
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

print("=" * 60)
print("Libraries Imported Successfully")
print("=" * 60)

Libraries Imported Successfully


## Project Configuration

This section defines standardized project paths used throughout the RetailPulse project.

Using `pathlib` ensures the notebook works consistently across Windows, macOS, and Linux without modifying file paths.

Directory structure used:

- data/raw/
- data/processed/
- models/
- reports/

In [2]:
# ==========================================================
# Project Configuration
# ==========================================================

# Current notebook directory
PROJECT_ROOT = Path.cwd().parent

# Project folders
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Create required directories
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Project directories configured successfully")
print("=" * 60)

print(f"Project Root      : {PROJECT_ROOT}")
print(f"Raw Data Folder   : {RAW_DATA_DIR}")
print(f"Processed Folder  : {PROCESSED_DATA_DIR}")
print(f"Models Folder     : {MODELS_DIR}")
print(f"Reports Folder    : {REPORTS_DIR}")

Project directories configured successfully
Project Root      : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics
Raw Data Folder   : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\data\raw
Processed Folder  : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\data\processed
Models Folder     : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\models
Reports Folder    : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\reports


## Load Online Retail II Dataset

The Online Retail II dataset contains two worksheets:

- Year 2009–2010
- Year 2010–2011

Both sheets will be merged into a single transactional dataset for subsequent preprocessing and analysis.

In [3]:
# ==========================================================
# Load Online Retail II Dataset
# ==========================================================

DATASET_FILE = RAW_DATA_DIR / "online_retail_II.xlsx"

if not DATASET_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found.\nPlease place the dataset here:\n{DATASET_FILE}"
    )

# Read both worksheets
df_2009 = pd.read_excel(DATASET_FILE, sheet_name="Year 2009-2010")
df_2010 = pd.read_excel(DATASET_FILE, sheet_name="Year 2010-2011")

# Merge datasets
raw_df = pd.concat([df_2009, df_2010], ignore_index=True)

print("=" * 60)
print("Dataset Loaded Successfully")
print("=" * 60)

print(f"2009-2010 Shape : {df_2009.shape}")
print(f"2010-2011 Shape : {df_2010.shape}")
print(f"Combined Shape  : {raw_df.shape}")

Dataset Loaded Successfully
2009-2010 Shape : (525461, 8)
2010-2011 Shape : (541910, 8)
Combined Shape  : (1067371, 8)


## Initial Dataset Inspection

Before cleaning the data, we perform an initial inspection to understand:

- Dataset dimensions
- Data types
- Missing values
- Duplicate records
- Basic statistics
- Memory usage

This assessment helps identify data quality issues before preprocessing.

In [4]:
# ==========================================================
# Initial Dataset Overview
# ==========================================================

print("=" * 70)
print("INITIAL DATASET OVERVIEW")
print("=" * 70)

print(f"\nDataset Shape : {raw_df.shape}")

print("\nColumn Names")
print("-" * 70)
print(raw_df.columns.tolist())

print("\nData Types")
print("-" * 70)
print(raw_df.dtypes)

print("\nMemory Usage")
print("-" * 70)

memory_mb = raw_df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"{memory_mb:.2f} MB")

print("\nFirst Five Records")
display(raw_df.head())

INITIAL DATASET OVERVIEW

Dataset Shape : (1067371, 8)

Column Names
----------------------------------------------------------------------
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data Types
----------------------------------------------------------------------
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

Memory Usage
----------------------------------------------------------------------
266.48 MB

First Five Records


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom


## Data Quality Assessment

The following checks are performed:

- Missing values
- Duplicate records
- Unique values
- Descriptive statistics

These metrics provide a quantitative understanding of the dataset quality before cleaning.

In [5]:
# ==========================================================
# Data Quality Assessment
# ==========================================================

print("=" * 70)
print("DATA QUALITY ASSESSMENT")
print("=" * 70)

# Missing Values
missing = raw_df.isnull().sum().to_frame("Missing Values")
missing["Missing (%)"] = (
    missing["Missing Values"] / len(raw_df) * 100
).round(2)

print("\nMissing Values")
display(missing)

# Duplicate Rows
duplicates = raw_df.duplicated().sum()

print(f"\nDuplicate Rows : {duplicates:,}")

# Unique Values
unique_df = pd.DataFrame({
    "Unique Values": raw_df.nunique()
})

print("\nUnique Values")
display(unique_df)

print("\nDescriptive Statistics (Numerical Columns)")
display(raw_df.describe().T)

print("\nDescriptive Statistics (Categorical Columns)")
display(raw_df.describe(include="object").T)

DATA QUALITY ASSESSMENT

Missing Values


,Missing Values,Missing (%)
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00



Duplicate Rows : 34,335

Unique Values


,Unique Values
Invoice,53628
StockCode,5305
Description,5698
Quantity,1057
InvoiceDate,47635
Price,2807
Customer ID,5942
Country,43



Descriptive Statistics (Numerical Columns)


,count,mean,min,25%,50%,75%,max,std
Quantity,1067371.00,9.94,-80995.00,1.00,3.00,10.00,80995.00,172.71
InvoiceDate,1067371,2011-01-02 21:13:55.394028544,2009-12-01 07:45:00,2010-07-09 09:46:00,2010-12-07 15:28:00,2011-07-22 10:23:00,2011-12-09 12:50:00,NaN
Price,1067371.00,4.65,-53594.36,1.25,2.10,4.15,38970.00,123.55
Customer ID,824364.00,15324.64,12346.00,13975.00,15255.00,16797.00,18287.00,1697.46



Descriptive Statistics (Categorical Columns)


,count,unique,top,freq
Invoice,1067371,53628,537434,1350
StockCode,1067371,5305,85123A,5829
Description,1062989,5698,WHITE HANGING HEART T-LIGHT HOLDER,5918
Country,1067371,43,United Kingdom,981330


## Data Cleaning Strategy

The raw retail dataset contains several data quality issues:

- Duplicate transactions
- Missing customer identifiers
- Missing product descriptions
- Cancelled invoices
- Invalid quantities
- Invalid prices

Cleaning decisions are documented explicitly to ensure reproducibility and transparency.

Cleaning workflow:

1. Standardize column names
2. Remove duplicate records
3. Handle missing values
4. Remove cancelled invoices
5. Remove invalid transactions
6. Convert data types

In [6]:
# ==========================================================
# Create Working Copy
# ==========================================================

df = raw_df.copy()

print("=" * 70)
print("Working copy created successfully")
print("=" * 70)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Working copy created successfully
Rows    : 1,067,371
Columns : 8


In [7]:
# ==========================================================
# Standardize Column Names
# ==========================================================

column_mapping = {
    "Invoice": "InvoiceID",
    "StockCode": "StockCode",
    "Description": "ProductDescription",
    "Quantity": "Quantity",
    "InvoiceDate": "InvoiceDate",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID",
    "Country": "Country"
}

df.rename(columns=column_mapping, inplace=True)

print("=" * 70)
print("Column names standardized")
print("=" * 70)

display(pd.DataFrame({
    "Original": raw_df.columns,
    "Standardized": df.columns
}))

Column names standardized


,Original,Standardized
0,Invoice,InvoiceID
1,StockCode,StockCode
2,Description,ProductDescription
3,Quantity,Quantity
4,InvoiceDate,InvoiceDate
5,Price,UnitPrice
6,Customer ID,CustomerID
7,Country,Country


In [8]:
# ==========================================================
# Duplicate Records
# ==========================================================

duplicate_count = df.duplicated().sum()

print("=" * 70)
print("Duplicate Records")
print("=" * 70)

print(f"Duplicate Rows Found : {duplicate_count:,}")

df.drop_duplicates(inplace=True)

print(f"Rows After Removal   : {len(df):,}")
print(f"Duplicates Removed   : {duplicate_count:,}")

Duplicate Records
Duplicate Rows Found : 34,335
Rows After Removal   : 1,033,036
Duplicates Removed   : 34,335


In [9]:
# ==========================================================
# Missing Values
# ==========================================================

missing_before = df.isnull().sum()

print("=" * 70)
print("Handling Missing Values")
print("=" * 70)

# Remove rows with missing product descriptions
df = df.dropna(subset=["ProductDescription"])

# Remove rows with missing Customer IDs
# (Required for customer analytics and ML notebooks)
df = df.dropna(subset=["CustomerID"])

missing_after = df.isnull().sum()

summary = pd.DataFrame({
    "Before": missing_before,
    "After": missing_after
})

display(summary)

Handling Missing Values


,Before,After
InvoiceID,0,0
StockCode,0,0
ProductDescription,4275,0
Quantity,0,0
InvoiceDate,0,0
UnitPrice,0,0
CustomerID,235151,0
Country,0,0


## Transaction Validation

Retail transaction datasets typically contain records that should not be used for customer analytics, including:

- Cancelled invoices
- Negative quantities
- Zero quantities
- Negative prices
- Zero prices

These records are removed to ensure that all downstream analyses are based on completed, valid purchase transactions.

In [10]:
# ==========================================================
# Cancelled Transactions
# ==========================================================

print("=" * 70)
print("Cancelled Transactions")
print("=" * 70)

cancelled = df["InvoiceID"].astype(str).str.startswith("C")

cancelled_count = cancelled.sum()

print(f"Cancelled Transactions : {cancelled_count:,}")

df = df.loc[~cancelled].copy()

print(f"Rows Remaining         : {len(df):,}")

Cancelled Transactions
Cancelled Transactions : 18,390
Rows Remaining         : 779,495


In [11]:
# ==========================================================
# Remove Invalid Quantity & Price
# ==========================================================

print("=" * 70)
print("Removing Invalid Transactions")
print("=" * 70)

negative_qty = (df["Quantity"] <= 0).sum()
negative_price = (df["UnitPrice"] <= 0).sum()

print(f"Invalid Quantity Rows : {negative_qty:,}")
print(f"Invalid Price Rows    : {negative_price:,}")

df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

print("\nRows Remaining")
print(f"{len(df):,}")

Removing Invalid Transactions
Invalid Quantity Rows : 0
Invalid Price Rows    : 70

Rows Remaining
779,425


In [12]:
# ==========================================================
# Reset Index
# ==========================================================

df.reset_index(drop=True, inplace=True)

print("=" * 70)
print("Dataset cleaned successfully")
print("=" * 70)

print(f"Final Shape : {df.shape}")

Dataset cleaned successfully
Final Shape : (779425, 8)


## Cleaning Summary

The dataset has now undergone the following preprocessing steps:

- Duplicate records removed
- Missing product descriptions removed
- Missing customer identifiers removed
- Cancelled invoices removed
- Invalid quantities removed
- Invalid prices removed

The resulting dataset represents valid customer purchase transactions and is ready for feature engineering.

# Feature Engineering

Feature engineering transforms cleaned transactional data into meaningful business attributes that improve exploratory analysis, machine learning models, and dashboard reporting.

The following features will be created:

## Monetary Features
- TotalAmount

## Calendar Features
- InvoiceYear
- InvoiceQuarter
- InvoiceMonth
- MonthName
- InvoiceWeek
- InvoiceDay
- DayName
- InvoiceHour
- IsWeekend

## Customer Features
- CustomerCountry

## Invoice Features
- InvoiceMonthYear

In [13]:
# ==========================================================
# Feature Engineering
# ==========================================================

print("=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)

# Monetary Feature
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

# Calendar Features
df["InvoiceYear"] = df["InvoiceDate"].dt.year
df["InvoiceQuarter"] = df["InvoiceDate"].dt.quarter
df["InvoiceMonth"] = df["InvoiceDate"].dt.month
df["MonthName"] = df["InvoiceDate"].dt.month_name()

# ISO week number
df["InvoiceWeek"] = (
    df["InvoiceDate"]
    .dt.isocalendar()
    .week
    .astype(int)
)

df["InvoiceDay"] = df["InvoiceDate"].dt.day

df["DayName"] = df["InvoiceDate"].dt.day_name()

df["InvoiceHour"] = df["InvoiceDate"].dt.hour

df["IsWeekend"] = (
    df["InvoiceDate"]
    .dt.dayofweek
    .isin([5, 6])
)

# Customer Feature
df["CustomerCountry"] = df["Country"]

# Invoice Month-Year
df["InvoiceMonthYear"] = (
    df["InvoiceDate"]
    .dt.to_period("M")
    .astype(str)
)

print("Feature engineering completed successfully.")

FEATURE ENGINEERING
Feature engineering completed successfully.


In [14]:
# ==========================================================
# Basket Level Features
# ==========================================================

basket_features = (
    df.groupby("InvoiceID")
      .agg(
          BasketSize=("StockCode", "count"),
          BasketValue=("TotalAmount", "sum")
      )
      .reset_index()
)

df = df.merge(
    basket_features,
    on="InvoiceID",
    how="left"
)

print("=" * 70)
print("Basket features created")
print("=" * 70)

print(f"Unique Invoices : {df['InvoiceID'].nunique():,}")

display(
    df[
        ["InvoiceID",
         "BasketSize",
         "BasketValue"]
    ].head()
)

Basket features created
Unique Invoices : 36,969


,InvoiceID,BasketSize,BasketValue
0,489434,8,505.30
1,489434,8,505.30
2,489434,8,505.30
3,489434,8,505.30
4,489434,8,505.30


In [15]:
# ==========================================================
# Verify Engineered Features
# ==========================================================

engineered_features = [
    "TotalAmount",
    "InvoiceYear",
    "InvoiceQuarter",
    "InvoiceMonth",
    "MonthName",
    "InvoiceWeek",
    "InvoiceDay",
    "DayName",
    "InvoiceHour",
    "IsWeekend",
    "CustomerCountry",
    "InvoiceMonthYear",
    "BasketSize",
    "BasketValue"
]

print("=" * 50)
print("ENGINEERED FEATURES")
print("=" * 50)

print(f"Total Features Created : {len(engineered_features)}")

for feature in engineered_features:
    print(f"✓ {feature}")

display(df.head())

ENGINEERED FEATURES
Total Features Created : 14
✓ TotalAmount
✓ InvoiceYear
✓ InvoiceQuarter
✓ InvoiceMonth
✓ MonthName
✓ InvoiceWeek
✓ InvoiceDay
✓ DayName
✓ InvoiceHour
✓ IsWeekend
✓ CustomerCountry
✓ InvoiceMonthYear
✓ BasketSize
✓ BasketValue


,InvoiceID,StockCode,ProductDescription,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,InvoiceYear,InvoiceQuarter,InvoiceMonth,MonthName,InvoiceWeek,InvoiceDay,DayName,InvoiceHour,IsWeekend,CustomerCountry,InvoiceMonthYear,BasketSize,BasketValue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom,83.40,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.30
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom,81.00,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.30
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom,81.00,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.30
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom,100.80,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.30
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom,30.00,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.30


## Feature Validation

Before exporting the processed dataset, we validate the engineered features to ensure:

- No missing values in engineered columns
- Correct data types
- Expected value distributions
- Dataset consistency

In [16]:
# ==========================================================
# Feature Validation
# ==========================================================

print("=" * 70)
print("FEATURE VALIDATION")
print("=" * 70)

# Dataset Shape
print(f"\nDataset Shape : {df.shape}")

# Missing Values
print("\nMissing Values in Engineered Features")
print("-" * 70)

engineered_cols = [
    "TotalAmount",
    "InvoiceYear",
    "InvoiceQuarter",
    "InvoiceMonth",
    "MonthName",
    "InvoiceWeek",
    "InvoiceDay",
    "DayName",
    "InvoiceHour",
    "IsWeekend",
    "CustomerCountry",
    "InvoiceMonthYear",
    "BasketSize",
    "BasketValue"
]

display(df[engineered_cols].isnull().sum())

print("\nData Types")
print("-" * 70)

display(df[engineered_cols].dtypes)

print("\nWeekend Distribution")
print("-" * 70)

display(df["IsWeekend"].value_counts())

print("\nPreview")
display(df.head())

FEATURE VALIDATION

Dataset Shape : (779425, 22)

Missing Values in Engineered Features
----------------------------------------------------------------------


TotalAmount         0
InvoiceYear         0
InvoiceQuarter      0
InvoiceMonth        0
MonthName           0
InvoiceWeek         0
InvoiceDay          0
DayName             0
InvoiceHour         0
IsWeekend           0
CustomerCountry     0
InvoiceMonthYear    0
BasketSize          0
BasketValue         0
dtype: int64


Data Types
----------------------------------------------------------------------


TotalAmount         float64
InvoiceYear           int32
InvoiceQuarter        int32
InvoiceMonth          int32
MonthName            object
InvoiceWeek           int64
InvoiceDay            int32
DayName              object
InvoiceHour           int32
IsWeekend              bool
CustomerCountry      object
InvoiceMonthYear     object
BasketSize            int64
BasketValue         float64
dtype: object


Weekend Distribution
----------------------------------------------------------------------


IsWeekend
False    648888
True     130537
Name: count, dtype: int64


Preview


,InvoiceID,StockCode,ProductDescription,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,InvoiceYear,InvoiceQuarter,InvoiceMonth,MonthName,InvoiceWeek,InvoiceDay,DayName,InvoiceHour,IsWeekend,CustomerCountry,InvoiceMonthYear,BasketSize,BasketValue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom,83.40,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.30
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom,81.00,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.30
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom,81.00,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.30
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom,100.80,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.30
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom,30.00,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.30


In [17]:
# ==========================================================
# Data Quality Report
# ==========================================================

quality_report = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Unique Customers",
        "Unique Products",
        "Unique Invoices",
        "Countries",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Value": [
        len(df),
        df.shape[1],
        df["CustomerID"].nunique(),
        df["StockCode"].nunique(),
        df["InvoiceID"].nunique(),
        df["Country"].nunique(),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum())
    ]
})

print("=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

display(quality_report)

DATA QUALITY REPORT


,Metric,Value
0,Total Rows,779425
1,Total Columns,22
2,Unique Customers,5878
3,Unique Products,4631
4,Unique Invoices,36969
5,Countries,41
6,Missing Values,0
7,Duplicate Rows,0


In [18]:
# ==========================================================
# Save Processed Dataset
# ==========================================================

retail_cleaned_path = PROCESSED_DATA_DIR / "retail_cleaned.csv"
analysis_data_path = PROCESSED_DATA_DIR / "analysis_data.csv"

df.to_csv(retail_cleaned_path, index=False)
df.to_csv(analysis_data_path, index=False)

print("=" * 70)
print("Datasets Saved Successfully")
print("=" * 70)

print(f"Retail Cleaned : {retail_cleaned_path}")
print(f"Analysis Data  : {analysis_data_path}")

Datasets Saved Successfully
Retail Cleaned : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\data\processed\retail_cleaned.csv
Analysis Data  : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\data\processed\analysis_data.csv


In [19]:
# ==========================================================
# Verify Saved Files
# ==========================================================

saved_df = pd.read_csv(
    retail_cleaned_path,
    parse_dates=["InvoiceDate"]
)

print("=" * 70)
print("VERIFICATION")
print("=" * 70)

print(f"Saved Shape : {saved_df.shape}")

assert saved_df.shape == df.shape

print("\nVerification Successful")
print("Notebook 01 Completed Successfully")

VERIFICATION
Saved Shape : (779425, 22)

Verification Successful
Notebook 01 Completed Successfully
